In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MultiLabelBinarizer, OneHotEncoder
import pickle

# -----------------------------
# 0️⃣ File paths
# -----------------------------
swiggy_path = r".\RESTAURANT_CLASSIFICATION\Data\swiggy.csv"
cleaned_path = r".\RESTAURANT_CLASSIFICATION\Data\cleaned_data.csv"
new_encoded_path = r".\RESTAURANT_CLASSIFICATION\Data\new\new_encoded_data.csv"
cuisine_encoder_path = r".\RESTAURANT_CLASSIFICATION\Data\cuisine_encoder.pkl"
city_encoder_path = r".\RESTAURANT_CLASSIFICATION\Data\city_encoder.pkl"

# -----------------------------
# 1️⃣ Load data
# -----------------------------
df = pd.read_csv(swiggy_path)

# Drop duplicates and missing rows
df.drop_duplicates(inplace=True)
df.dropna(inplace=True)

# -----------------------------
# 2️⃣ Clean numeric columns
# -----------------------------
# rating
df['rating'] = df['rating'].replace('--', np.nan).astype(float)
df['rating'].fillna(df['rating'].mean(), inplace=True)

# rating_count
def clean_rating_count(x):
    if pd.isna(x):
        return 0
    x = str(x).strip()
    if x == "Too Few Ratings":
        return 0
    x = x.replace("+", "").replace(" ratings", "")
    try:
        return int(x)
    except:
        return 0
df['rating_count'] = df['rating_count'].apply(clean_rating_count)

# cost
df['cost'] = df['cost'].str.replace('₹','', regex=True)
df['cost'] = df['cost'].str.replace(',','', regex=True)
df['cost'] = pd.to_numeric(df['cost'], errors='coerce')

# Reset index after cleaning
df.reset_index(drop=True, inplace=True)

# Save cleaned numeric data
df.to_csv(cleaned_path, index=False)

# -----------------------------
# 3️⃣ MultiLabelBinarizer for cuisine
# -----------------------------
df['cuisine'] = df['cuisine'].str.split(",")
mlb = MultiLabelBinarizer()
cuisine_encoded = pd.DataFrame(
    mlb.fit_transform(df['cuisine']),
    columns=mlb.classes_
)

# Reset index to match df
cuisine_encoded.reset_index(drop=True, inplace=True)

# Save cuisine encoder
with open(cuisine_encoder_path, "wb") as f:
    pickle.dump(mlb, f)

# -----------------------------
# 4️⃣ OneHotEncoder for city
# -----------------------------
ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
city_encoded = pd.DataFrame(
    ohe.fit_transform(df[['city']]),
    columns=ohe.get_feature_names_out(['city'])
)

# Reset index to match df
city_encoded.reset_index(drop=True, inplace=True)

# Save city encoder
with open(city_encoder_path, "wb") as f:
    pickle.dump(ohe, f)

# -----------------------------
# 5️⃣ Combine numeric + encoded columns
# -----------------------------
numeric_cols = ['rating', 'rating_count', 'cost']

df_cluster = pd.concat([df[numeric_cols], cuisine_encoded, city_encoded], axis=1)

df_cluster.reset_index(drop=True, inplace=True)

df_cluster.to_csv(new_encoded_path, index=False)

# -----------------------------
# 6️⃣ Save final encoded CSV
# -----------------------------

print("Saving to:", new_encoded_path)
print(df_cluster.head())
print(df_cluster.dtypes)
df_cluster.to_csv(new_encoded_path, index=False)

print("✅ Encoding complete! Cleaned numeric + cuisine + city columns saved successfully.")

C:\Users\ajith\AppData\Local\Temp\ipykernel_11620\297703302.py:29: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['rating'].fillna(df['rating'].mean(), inplace=True)


Saving to: C:\Users\ajith\GUVI_AI_ML\PROJECTS\RESTAURANT_CLASSIFICATION\Data\new\new_encoded_data.csv
     rating  rating_count  cost  8:15 To 11:30 Pm  Afghani  African  American  \
0  3.894513             0   200                 0        0        0         0   
1  4.400000            50   200                 0        0        0         0   
2  3.800000           100   100                 0        0        0         0   
3  3.700000            20   250                 0        0        0         0   
4  3.894513             0   250                 0        0        0         0   

   Andhra  Arabian  Asian  ...  city_Washim  city_West Chd,Chandigarh  \
0       0        0      0  ...          0.0                       0.0   
1       0        0      0  ...          0.0                       0.0   
2       0        0      0  ...          0.0                       0.0   
3       0        0      0  ...          0.0                       0.0   
4       0        0      0  ...          0.0   